# 🔄 Preprocesamiento de Datos

En este notebook se realiza la limpieza y preparación del dataset. Los pasos de preprocesamiento incluyen:
1. Filtrado de registros inconsistentes.
2. Imputación de valores nulos (`bmi`).
3. Codificación de variables categóricas (Encoding).
4. División en conjuntos de entrenamiento (80%) y prueba (20%).
5. Estandarización de variables numéricas (Scaling).
6. Balanceo de clases en entrenamiento mediante **SMOTE**.

In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
import os

print("Librerías importadas correctamente.")

Librerías importadas correctamente.


### 1. Carga de Datos y Limpieza de Faltantes/Inconsistencias

In [4]:
# Cargar datos
df = pd.read_csv('../data/raw/healthcare-dataset-stroke-data.csv')
df['bmi'] = pd.to_numeric(df['bmi'], errors='coerce')

# Remover el único registro con género 'Other' para evitar problemas en codificación binaria
df = df[df['gender'] != 'Other']
print(f"Dimensiones después de limpieza inicial: {df.shape}")

Dimensiones después de limpieza inicial: (5109, 12)


#### Imputación del BMI
Utilizaremos la mediana de BMI agrupada por género e hipertensión para imputar de forma más precisa los valores nulos.

In [5]:
# Calcular la mediana según género e hipertensión
imputation_values = df.groupby(['gender', 'hypertension'])['bmi'].transform('median')
df['bmi'] = df['bmi'].fillna(imputation_values)
print(f"Valores faltantes restantes en bmi: {df['bmi'].isnull().sum()}")

Valores faltantes restantes en bmi: 0


#### Guardar Dataset Limpio
Guardamos el CSV limpio en `data/processed/stroke_data_clean.csv` tal como se solicita en los requerimientos.

In [6]:
# Crear carpeta processed si no existe
os.makedirs('../data/processed', exist_ok=True)

# Guardar dataset limpio
df.to_csv('../data/processed/stroke_data_clean.csv', index=False)
print("Dataset limpio guardado como 'data/processed/stroke_data_clean.csv'")

Dataset limpio guardado como 'data/processed/stroke_data_clean.csv'


### 2. Codificación de Variables (Encoding)

In [7]:
# Codificación binaria para variables con dos categorías
df['gender'] = df['gender'].map({'Male': 1, 'Female': 0})
df['ever_married'] = df['ever_married'].map({'Yes': 1, 'No': 0})
df['Residence_type'] = df['Residence_type'].map({'Urban': 1, 'Rural': 0})

# Codificación One-Hot para variables categóricas múltiples
df = pd.get_dummies(df, columns=['work_type', 'smoking_status'], drop_first=True, dtype=int)

# Eliminar la columna ID que no tiene valor predictivo
df = df.drop(columns=['id'])

print(f"Columnas después de codificación: {df.shape[1]}")
df.head()

Columnas después de codificación: 16


,gender,age,hypertension,heart_disease,ever_married,Residence_type,avg_glucose_level,bmi,stroke,work_type_Never_worked,work_type_Private,work_type_Self-employed,work_type_children,smoking_status_formerly smoked,smoking_status_never smoked,smoking_status_smokes
0,1,67.0,0,1,1,1,228.69,36.6,1,0,1,0,0,1,0,0
1,0,61.0,0,0,1,0,202.21,27.5,1,0,0,1,0,0,1,0
2,1,80.0,0,1,1,0,105.92,32.5,1,0,1,0,0,0,1,0
3,0,49.0,0,0,1,1,171.23,34.4,1,0,1,0,0,0,0,1
4,0,79.0,1,0,1,0,174.12,24.0,1,0,0,1,0,0,1,0


### 3. División en Train / Test

In [8]:
X = df.drop(columns=['stroke'])
y = df['stroke']

# División del dataset (80% entrenamiento, 20% prueba, estratificado por variable objetivo)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Entrenamiento: {X_train.shape[0]} registros")
print(f"Prueba: {X_test.shape[0]} registros")

Entrenamiento: 4087 registros
Prueba: 1022 registros


### 4. Estandarización de Variables Numéricas (Scaling)

In [9]:
num_cols = ['age', 'avg_glucose_level', 'bmi']
scaler = StandardScaler()

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

# Ajustar scaler en train y transformar train y test
X_train_scaled[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test_scaled[num_cols] = scaler.transform(X_test[num_cols])

X_train_scaled.head()

,gender,age,hypertension,heart_disease,ever_married,Residence_type,avg_glucose_level,bmi,work_type_Never_worked,work_type_Private,work_type_Self-employed,work_type_children,smoking_status_formerly smoked,smoking_status_never smoked,smoking_status_smokes
845,0,0.209397,0,0,1,1,-0.821221,0.546315,0,1,0,0,0,1,0
3745,0,-0.629845,0,0,0,1,-0.485884,-0.990610,0,1,0,0,0,1,0
4184,0,-0.364822,0,0,1,0,0.302317,-0.771049,0,1,0,0,0,1,0
3410,1,-0.232310,0,0,1,0,0.062342,0.494654,0,1,0,0,0,1,0
284,1,-1.292405,0,0,0,1,-0.527297,0.352585,0,0,0,0,0,0,0


### 5. Balanceo de Clases mediante SMOTE

Para evitar que la red neuronal aprenda sesgada hacia la clase mayoritaria (no stroke), aplicamos SMOTE en el conjunto de entrenamiento.

In [10]:
print(f"Distribución previa a SMOTE:\n{y_train.value_counts()}")

smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train_scaled, y_train)

print(f"Distribución posterior a SMOTE:\n{y_train_res.value_counts()}")

Distribución previa a SMOTE:
stroke
0    3888
1     199
Name: count, dtype: int64
Distribución posterior a SMOTE:
stroke
0    3888
1    3888
Name: count, dtype: int64


#### Guardar Variables Procesadas
Guardamos las matrices de entrenamiento y prueba escaladas y balanceadas para modelar directamente.

In [ ]:
import pandas as pd

# Crear DataFrame de entrenamiento
train_df = pd.DataFrame(X_train_res, columns=X.columns)
train_df['stroke'] = y_train_res.values

# Crear DataFrame de prueba
test_df = pd.DataFrame(X_test_scaled, columns=X.columns)
test_df['stroke'] = y_test.values

# Guardar en un solo archivo Excel con dos hojas
with pd.ExcelWriter('../data/processed/model_ready_data.xlsx') as writer:
    train_df.to_excel(writer, sheet_name='Train_Data', index=False)
    test_df.to_excel(writer, sheet_name='Test_Data', index=False)

print("Datos de entrenamiento y prueba exportados en '../data/processed/model_ready_data.xlsx'")

In [14]:
np.savez('../data/processed/model_ready_data.npz', 
         X_train=X_train_res.values, y_train=y_train_res.values,
         X_test=X_test_scaled.values, y_test=y_test.values,
         feature_names=X.columns.values)

print("Datos de entrenamiento y prueba exportados en 'data/processed/model_ready_data.npz'")

Datos de entrenamiento y prueba exportados en 'data/processed/model_ready_data.npz'
